# Disc Detection

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

In [ ]:
%matplotlib widget

## Imports

In [ ]:
from pathlib import Path

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

import panel as pn

import enderleaf.const as ec
from enderleaf.draw import image_grid
from enderleaf.tools import read_dataframe, write_dataframe
from enderleaf.image import load_image, to_pil, canny, find_circles, filter_circles

## Constants

In [ ]:
PATH_TO_DATA = Path(".").joinpath("output", "job_data", "Exp26DM14", "I2")
PATH_TO_IMAGES = Path(".").joinpath("output", "images", "Exp26DM14", "I2")
# FACTOR = 2
MAX_CIRCLES=3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")]).sort_values(
    ["plate", "row", "col"]
)
df

In [ ]:
pn.extension("ipywidgets")

In [ ]:
sel_color_space = pn.widgets.Select(
    name="Color space", options=list(ec.COLOR_SPACES.keys()), value="hsv", width=100
)
sel_channel = pn.widgets.Select(
    name="Channel", options=ec.COLOR_SPACES[sel_color_space.value], value="s", width=100
)
ii_factor = pn.widgets.IntInput(
    name="Resize factor", start=1, end=20, step=1, value=4, width=100
)
ii_median_blur = pn.widgets.IntInput(
    name="Median blur", start=1, end=21, step=2, value=7, width=100
)
ii_gaussian_blur = pn.widgets.IntInput(
    name="Gaussian blur", start=1, end=21, step=2, value=1, width=100
)
chk_normalize = pn.widgets.Checkbox(name="Normalize", value=True)
irs_threshold = pn.widgets.IntRangeSlider(
    name="Thresholds",
    start=0,
    end=255,
    step=1,
    value=(150, 255),
    width=200,
)
sel_plate = pn.widgets.Select(name="Plate", options=list(df.plate.unique()), width=100)
sel_row = pn.widgets.Select(name="Row", options=list(df.row.unique()), width=100)
sel_col = pn.widgets.Select(name="Col", options=list(df.col.unique()), width=100)

dt_accepted = pn.pane.DataFrame(sizing_mode="stretch_width")
dt_discarded_position = pn.pane.DataFrame(sizing_mode="stretch_width")
dt_discarded_accu = pn.pane.DataFrame(sizing_mode="stretch_width")

bt_random = pn.widgets.Button(name="Random Disc")


def on_random(event):
    _df = df[["plate", "row", "col"]].drop_duplicates()
    row = df.sample(n=1).iloc[0]
    sel_plate.value = row.plate
    sel_row.value = row.row
    sel_col.value = row.col


bt_random.on_click(on_random)


def filter_df() -> pd.DataFrame:
    return df[
        (df.plate == sel_plate.value)
        & (df.col == sel_col.value)
        & (df.row == sel_row.value)
    ]


ds_index = pn.widgets.DiscreteSlider(
    name="Timestamp",
    options=filter_df().sort_values("date_time").date_time.to_list(),
    sizing_mode="stretch_width",
)

img_transformed = pn.pane.Image(sizing_mode="stretch_width")
img_edges = pn.pane.Image(sizing_mode="stretch_width")
img_circles = pn.pane.Image(sizing_mode="stretch_width")


def get_data(plate, row, col, timestamp):
    return df[
        (df.plate == plate)
        & (df.col == col)
        & (df.row == row)
        & (df.date_time == timestamp)
    ].iloc[0]


def get_circles(
    row, channel, thresholds, factor, normalize, median_blur, gaussian_blur
):
    image = load(row)
    im_width = image.shape[1] // factor
    im_height = image.shape[0] // factor
    image = cv2.resize(image, (im_width, im_height))
    if normalize is True:
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    if median_blur > 1:
        image = cv2.medianBlur(image, ksize=median_blur)
    if gaussian_blur > 1:
        image = cv2.GaussianBlur(image, ksize=(gaussian_blur, gaussian_blur), sigmaX=0)
    img_transformed.object = to_pil(image)
    edges = canny(
        image=image,
        color_space=sel_color_space.value,
        channel=channel,
        min_thresholf=thresholds[0],
        max_threshold=thresholds[1],
    )
    img_edges.object = to_pil(edges)

    return filter_circles(
        find_circles(
            edges=edges,
            radii=np.arange(450 // factor, 550 // factor, 20 // factor),
            max_circles=MAX_CIRCLES,
        ),
        img_width=im_width,
        img_height=im_height,
    )


@pn.depends(sel_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_channel.options = ec.COLOR_SPACES[cs]
    sel_channel.value = ec.COLOR_SPACES[cs][0]


@pn.depends(sel_plate.param.value, watch=True)
def on_plate_changed(plate):
    sel_row.value = sel_row.options[0]
    sel_col.value = sel_col.options[0]
    ds_index.value = ds_index.options[0]


@pn.depends(sel_col.param.value, sel_row.param.value, watch=True)
def on_pos_changed(col, row):
    _df = filter_df().sort_values("date_time")
    ds_index.options = _df.date_time.to_list()
    ds_index.value = ds_index.options[0]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_channel,
            ds_index,
            irs_threshold,
            ii_factor,
            chk_normalize,
            ii_median_blur,
            ii_gaussian_blur,
        ]
    ],
    watch=True,
)
def on_channel_selected(
    channel, index, thresholds, factor, normalize, median_blur, gaussian_blur
):
    row = get_data(
        plate=sel_plate.value, row=sel_row.value, col=sel_col.value, timestamp=index
    )
    circles = get_circles(
        row=row,
        channel=channel,
        thresholds=thresholds,
        factor=factor,
        normalize=normalize,
        median_blur=median_blur,
        gaussian_blur=gaussian_blur,
    )

    dt_accepted.object = pd.DataFrame(
        circles["accepted"], columns=["accu", "cx", "cy", "r"]
    )
    dt_discarded_position.object = pd.DataFrame(
        circles["discarded_position"], columns=["accu", "cx", "cy", "r"]
    )
    dt_discarded_accu.object = pd.DataFrame(
        circles["discarded_accu"], columns=["accu", "cx", "cy", "r"]
    )

    out = load(row)
    im_width = out.shape[1] // factor
    im_height = out.shape[0] // factor
    out = cv2.resize(out, (im_width, im_height))
    for accu, cx, cy, r in circles["accepted"]:
        out = cv2.circle(out, (cx, cy), r, (0, 0, 0), 4)
    for accu, cx, cy, r in circles["discarded_position"]:
        out = cv2.circle(out, (cx, cy), r, (255, 0, 0), 2)
    for accu, cx, cy, r in circles["discarded_accu"]:
        out = cv2.circle(out, (cx, cy), r, (255, 0, 255), 2)
    img_circles.object = to_pil(out)


on_channel_selected(
    sel_channel.value,
    ds_index.value,
    irs_threshold.value,
    ii_factor.value,
    chk_normalize.value,
    gaussian_blur=ii_gaussian_blur.value,
    median_blur=ii_median_blur.value,
)

pn.Column(
    pn.layout.FlexBox(
        sel_color_space,
        sel_channel,
        chk_normalize,
        sel_plate,
        sel_row,
        sel_col,
        bt_random,
        ii_median_blur,
        ii_gaussian_blur,
        ii_factor,
        irs_threshold,
    ),
    pn.Row(
        pn.layout.WidgetBox("### Accepted", dt_accepted),
        pn.layout.WidgetBox("### Discarded position", dt_discarded_position),
        pn.layout.WidgetBox("### Discarded accu", dt_discarded_accu),
    ),
    ds_index,
    pn.Row(img_transformed, img_edges, img_circles),
)

In [ ]:
row = get_data(plate=sel_plate.value, row=sel_row.value, col=sel_col.value, timestamp=ds_index.value)
data = get_circles(
    row,
    sel_channel.value,
    irs_threshold.value,
    ii_factor.value,
    chk_normalize.value,
    gaussian_blur=ii_gaussian_blur.value,
    median_blur=ii_median_blur.value,
)
data

In [ ]:
def get_row_df(row) -> pd.DataFrame:
    return pd.DataFrame(row).T.reset_index(drop=True)


def get_circle_df(circles, key) -> pd.DataFrame:
    return (
        pd.DataFrame(circles[key], columns=["accu", "cx", "cy", "radius"])
        .assign(kind=key)
        .reset_index(drop=True)
    )


def merge_row_circles(row):
    circles = get_circles(
        row,
        sel_channel.value,
        irs_threshold.value,
        ii_factor.value,
        chk_normalize.value,
        gaussian_blur=ii_gaussian_blur.value,
        median_blur=ii_median_blur.value,
    )
    df_row = get_row_df(row)
    return pd.concat(
        [
            pd.concat([df_row, get_circle_df(circles, key)], axis=1)
            for key in circles.keys()
            if len(circles[key]) > 0
        ]
    )

In [ ]:
path_to_file = PATH_TO_DATA.parent.joinpath("circles").with_suffix(".csv")
if path_to_file.is_file() is False:
    df_circles = pd.concat([merge_row_circles(r) for i, r in tqdm(list(df.iterrows()))])
    write_dataframe(df_circles, path_to_file)
else:
    df_circles = read_dataframe(path_to_file)